# 08 — Grid Viability and Friction Points

For each proposed station (NB07), find the nearest electrical substation using BallTree
nearest-neighbor search (assumption G3). Assign `grid_status` based on available capacity
thresholds (D1–D3). Generate friction points list (Moderate + Congested only).

**Key insight:** ~80% of Spain's 2,137 unique substations show 0 MW available capacity —
authentic grid saturation. Most proposed stations will be friction points.
This is the central strategic finding for Iberdrola.

## Data Inputs
- `data/processed/proposed_stations.csv` — from NB07
- `data/processed/grid_consolidated.csv` — 2,137 deduplicated substations (3 DSOs)

## Data Outputs
- `data/processed/stations_with_grid_status.csv` — all proposed stations + grid info
- `data/processed/friction_points.csv` — Moderate + Congested only (File_3 source)

In [12]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path

if os.path.basename(os.getcwd()) == 'notebooks':
    sys.path.insert(0, os.path.dirname(os.getcwd()))
    DATA_DIR = Path('../data/processed')
else:
    sys.path.insert(0, os.getcwd())
    DATA_DIR = Path('data/processed')

from src.constants import (
    POWER_PER_CHARGER_KW,
    MAX_SUBSTATION_SEARCH_RADIUS_KM,
    SUBSTATION_DIST_OPTIMAL_KM,
    SUBSTATION_DIST_FEASIBLE_KM,
    DEFAULT_STATUS_IF_NO_SUBSTATION,
    VALID_GRID_STATUSES_FILE3,
    VALID_DISTRIBUTORS,
    GRID_SUFFICIENT_MIN_MW, GRID_MODERATE_MIN_MW,
)
from src.grid_analysis import classify_grid_status, is_friction_point
from src.geo_utils import find_nearest_substation

print('✅ Imports OK')
print(f'   Grid thresholds: Sufficient ≥{GRID_SUFFICIENT_MIN_MW} MW | Moderate ≥{GRID_MODERATE_MIN_MW} MW | Congested <{GRID_MODERATE_MIN_MW} MW')
print(f'   Substation search radius: {MAX_SUBSTATION_SEARCH_RADIUS_KM} km')
print(f'   Power per charger: {POWER_PER_CHARGER_KW} kW')

✅ Imports OK
   Grid thresholds: Sufficient ≥5.0 MW | Moderate ≥1.0 MW | Congested <1.0 MW
   Substation search radius: 25 km
   Power per charger: 150 kW


## Step 1: Load inputs

In [13]:
# Proposed stations from NB07
stations = pd.read_csv(DATA_DIR / 'proposed_stations.csv')
print(f'📍 Proposed stations: {len(stations):,}')
if len(stations) == 0:
    print('⚠️  No proposed stations — check NB07 output')

# Grid capacity — deduplicated to physical substations (NB05 output)
grid = pd.read_csv(DATA_DIR / 'grid_consolidated.csv')
print(f'⚡ Substations (deduplicated): {len(grid):,}')
print(f'   DSO breakdown:')
for dso, count in grid['distributor_network'].value_counts().items():
    zero_pct = (grid[grid['distributor_network']==dso]['available_capacity_mw'] == 0).mean() * 100
    print(f'   {dso}: {count:,} substations ({zero_pct:.0f}% at 0 MW available)')

📍 Proposed stations: 9
⚡ Substations (deduplicated): 2,137
   DSO breakdown:
   i-DE: 1,061 substations (88% at 0 MW available)
   Endesa: 981 substations (78% at 0 MW available)
   Viesgo: 95 substations (24% at 0 MW available)


## Step 2: Match Each Station to Nearest Substation

BallTree haversine nearest-neighbor search (G3).  
Connection tiers (D4): optimal ≤5 km, feasible 5–15 km, high-cost 15–25 km.

In [14]:
results = []

for _, row in stations.iterrows():
    match = find_nearest_substation(
        station_lat=row['latitude'],
        station_lon=row['longitude'],
        substations_df=grid,
        max_radius_km=MAX_SUBSTATION_SEARCH_RADIUS_KM,
    )
    if match:
        results.append({
            'location_id': row['location_id'],
            'available_capacity_mw': match['available_capacity_mw'],
            'distributor_network': match['distributor_network'],
            'connection_distance_km': match['distance_km'],
            'connection_tier': match['connection_tier'],
        })
    else:
        results.append({
            'location_id': row['location_id'],
            'available_capacity_mw': 0.0,
            'distributor_network': 'Unknown',
            'connection_distance_km': None,
            'connection_tier': 'none',
        })

grid_results = pd.DataFrame(results)
stations_grid = stations.merge(grid_results, on='location_id', how='left')

matched = grid_results['distributor_network'].ne('Unknown').sum()
unmatched = len(stations) - matched
print(f'⚡ Substation matching complete:')
print(f'   Stations matched: {matched:,} / {len(stations):,} ({matched/max(len(stations),1)*100:.1f}%)')
if len(grid_results) > 0:
    tier_counts = grid_results['connection_tier'].value_counts()
    for tier, count in tier_counts.items():
        labels = {'optimal': '≤5 km (optimal)', 'feasible': '5-15 km (feasible)',
                  'high_cost': '15-25 km (high-cost)', 'none': f'>25 km (no substation within {MAX_SUBSTATION_SEARCH_RADIUS_KM} km)'}
        print(f'   {labels.get(tier, tier)}: {count:,}')

if unmatched > 0:
    no_match = stations_grid[stations_grid['connection_tier'] == 'none']
    print(f'\n   ⚠️  {unmatched} station(s) have no substation within {MAX_SUBSTATION_SEARCH_RADIUS_KM} km:')
    for _, r in no_match.iterrows():
        print(f'      {r["location_id"]} ({r["route_segment"]}) @ ({r["latitude"]:.3f}, {r["longitude"]:.3f})')
    print(f'   → These require new grid infrastructure (not just a connection extension).')
    print(f'   → Classified as Congested (worst-case assumption per D1).')

⚡ Substation matching complete:
   Stations matched: 7 / 9 (77.8%)
   15-25 km (high-cost): 3
   5-15 km (feasible): 3
   >25 km (no substation within 25 km): 2
   ≤5 km (optimal): 1

   ⚠️  2 station(s) have no substation within 25 km:
      STA_0003 (N-330) @ (39.936, -1.298)
      STA_0009 (AP-9) @ (42.682, -8.635)
   → These require new grid infrastructure (not just a connection extension).
   → Classified as Congested (worst-case assumption per D1).


## Step 3: Classify Grid Status & Compute Estimated Demand

In [15]:
# Classify grid_status using available capacity
stations_grid['grid_status'] = stations_grid['available_capacity_mw'].apply(
    classify_grid_status
)

# estimated_demand_kw = n_chargers × 150 kW (mandatory formula, File_3)
stations_grid['estimated_demand_kw'] = (
    stations_grid['n_chargers_proposed'] * POWER_PER_CHARGER_KW
)

# Add MW equivalent for apples-to-apples comparison with available_capacity_mw
stations_grid['estimated_demand_mw'] = stations_grid['estimated_demand_kw'] / 1000

print('⚡ Grid status distribution:')
for status, count in stations_grid['grid_status'].value_counts().items():
    pct = count / len(stations_grid) * 100 if len(stations_grid) > 0 else 0
    print(f'   {status}: {count:,} ({pct:.1f}%)')

print(f'\n📊 Demand vs available capacity (same units):')
print(f'   {"Station":<12} {"Demand (MW)":>12} {"Available (MW)":>15} {"Deficit (MW)":>13} {"Status":<12}')
print(f'   {"─"*12:<12} {"─"*12:>12} {"─"*15:>15} {"─"*13:>13} {"─"*12:<12}')
for _, r in stations_grid.iterrows():
    deficit = r['available_capacity_mw'] - r['estimated_demand_mw']
    print(f'   {r["location_id"]:<12} {r["estimated_demand_mw"]:>12.2f} {r["available_capacity_mw"]:>15.2f} {deficit:>+13.2f} {r["grid_status"]:<12}')

print(f'\n   This reflects Spain\'s authentic grid saturation:')
print(f'   ~80% of 2,137 unique substations show 0 MW available (not a data error, see G1)')

⚡ Grid status distribution:
   Congested: 9 (100.0%)

📊 Demand vs available capacity (same units):
   Station       Demand (MW)  Available (MW)  Deficit (MW) Status      
   ──────────── ──────────── ─────────────── ───────────── ────────────
   STA_0001             0.60            0.00         -0.60 Congested   
   STA_0002             0.60            0.00         -0.60 Congested   
   STA_0003             0.60            0.00         -0.60 Congested   
   STA_0004             0.60            0.00         -0.60 Congested   
   STA_0005             0.60            0.00         -0.60 Congested   
   STA_0006             0.60            0.00         -0.60 Congested   
   STA_0007             0.30            0.00         -0.30 Congested   
   STA_0008             0.30            0.06         -0.24 Congested   
   STA_0009             0.60            0.00         -0.60 Congested   

   This reflects Spain's authentic grid saturation:
   ~80% of 2,137 unique substations show 0 MW available 

## Step 4: Extract Friction Points & Validate

In [16]:
# Friction points = Moderate + Congested (never Sufficient)
friction = stations_grid[stations_grid['grid_status'].isin(VALID_GRID_STATUSES_FILE3)].copy()

print(f'🔥 Friction points: {len(friction):,} / {len(stations_grid):,} stations')
print(f'   (Moderate + Congested only — Sufficient excluded by brief rule)')

# Validation
assert not friction['grid_status'].isin(['Sufficient']).any(), \
    'ERROR: Sufficient status found in friction points!'
assert (stations_grid['estimated_demand_kw'] == stations_grid['n_chargers_proposed'] * 150).all(), \
    'ERROR: estimated_demand_kw ≠ n_chargers × 150'

print('✅ Validation passed: no Sufficient in friction, estimated_demand_kw formula OK')

# Drop display-only column before saving
save_cols = [c for c in stations_grid.columns if c != 'estimated_demand_mw']

# Save stations_with_grid_status
out_stations = DATA_DIR / 'stations_with_grid_status.csv'
stations_grid[save_cols].to_csv(out_stations, index=False)
print(f'\n💾 stations_with_grid_status.csv → {len(stations_grid):,} rows')
print(f'   Columns: {save_cols}')

# Save friction points
out_friction = DATA_DIR / 'friction_points.csv'
friction[save_cols].to_csv(out_friction, index=False)
print(f'💾 friction_points.csv → {len(friction):,} rows')
stations_grid[save_cols].head(9)

🔥 Friction points: 9 / 9 stations
   (Moderate + Congested only — Sufficient excluded by brief rule)
✅ Validation passed: no Sufficient in friction, estimated_demand_kw formula OK

💾 stations_with_grid_status.csv → 9 rows
   Columns: ['location_id', 'latitude', 'longitude', 'route_segment', 'n_chargers_proposed', 'available_capacity_mw', 'distributor_network', 'connection_distance_km', 'connection_tier', 'grid_status', 'estimated_demand_kw']
💾 friction_points.csv → 9 rows


,location_id,latitude,longitude,route_segment,n_chargers_proposed,available_capacity_mw,distributor_network,connection_distance_km,connection_tier,grid_status,estimated_demand_kw
0,STA_0001,38.775545,-2.446657,N-322,4,0.00,i-DE,20.605,high_cost,Congested,600
1,STA_0002,37.900207,-6.644477,N-433,4,0.00,Endesa,10.256,feasible,Congested,600
2,STA_0003,39.935600,-1.297858,N-330,4,0.00,Unknown,NaN,none,Congested,600
3,STA_0004,41.512416,-0.068422,AP-2,4,0.00,Endesa,6.556,feasible,Congested,600
4,STA_0005,41.495475,-0.183176,N-2,4,0.00,Endesa,16.008,high_cost,Congested,600
5,STA_0006,40.455491,-1.232499,A-23,4,0.00,Endesa,0.722,optimal,Congested,600
6,STA_0007,37.823960,-6.716242,N-435,2,0.00,Endesa,12.746,feasible,Congested,300
7,STA_0008,43.069502,-4.755363,N-621,2,0.06,Viesgo,15.926,high_cost,Congested,300
8,STA_0009,42.681726,-8.634557,AP-9,4,0.00,Unknown,NaN,none,Congested,600
